Pipeline de extração de features dos Protocolos de Exercício, AEROBIC e ANAEROBIC.

Processa os sinais fisiológicos dos protocolos de exercício do artigo/dataset, cada protocolo gera um CSV separado com features das fases:

 - Exercício:                 (label=1)
 - Repouso/baseline/cooldown: (label=0)

Protocolos:
    - ANAEROBIC V1 (Sxx): 3 sprints de 30s (Wingate adaptado)
    - ANAEROBIC V2 (Fxx): 4 sprints de 45s
    - AEROBIC V1 (Sxx)  : 10 estágios de ciclismo (60-110 rpm, Storer-Davis adaptado)
    - AEROBIC V2 (Fxx)  : 4 estágios de ciclismo (70-95 rpm, simplificado)

Para o Cenário 2 (AEROBIC vs ANAEROBIC), a combinação dos datasets é feita
no script de modelagem, usando apenas as janelas de exercício (label=1).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import sys
PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore', message='Mean of empty slice')

from src.config import RAW_DATA_DIR, PROCESSED_DIR, WINDOW_SIZE, STEP_SIZE
from src.data_loader import load_subject_raw_data
from src.preprocessing import SignalPreprocessor
from src.labeling import get_session_start_time, load_tags, apply_labels
from src.feature_extraction import FeatureExtractor

As exclusões são baseadas no 'data_constraints.txt' do artigo/dataset:

AEROBIC:
- S03 -> encerrou protocolo em 90 rpm (incompleto)
- S07 -> encerrou protocolo em 95 rpm (incompleto)
- S11 -> conexão Bluetooth perdida (arquivos split: S11_a, S11_b)
- S12 -> não realizou protocolo aeróbico (sem dados)

ANAEROBIC:
- S06 -> encerrou protocolo antes (último sprint faltando)
- S16 -> conexão Bluetooth perdida (arquivos split: S16_a, S16_b)

In [ ]:
PROTOCOLS = ['ANAEROBIC', 'AEROBIC']

EXCLUSIONS = {
    'ANAEROBIC': ['S06', 'S16', 'S16_a', 'S16_b'],
    'AEROBIC':   ['S03', 'S07', 'S11', 'S11_a', 'S11_b', 'S12'],
}

for protocol in PROTOCOLS:
    protocol_dir = RAW_DATA_DIR / protocol
    if not protocol_dir.exists():
        print(f"AVISO: {protocol_dir} não encontrado")
        continue
    
    subjects = sorted([p.name for p in protocol_dir.iterdir() if p.is_dir()])
    valid = [s for s in subjects if s not in EXCLUSIONS.get(protocol, [])]
    print(f"{protocol}: {len(subjects)} total | {len(valid)} válidos | Excluídos: {EXCLUSIONS[protocol]}")
    print(f"  Sujeitos: {valid}")

ANAEROBIC: 32 total | 29 válidos | Excluídos: ['S06', 'S16', 'S16_a', 'S16_b']
  Sujeitos: ['S01', 'S02', 'S03', 'S04', 'S05', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13', 'S14', 'S15', 'S17', 'S18', 'f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13']
AEROBIC: 31 total | 27 válidos | Excluídos: ['S03', 'S07', 'S11', 'S11_a', 'S11_b', 'S12']
  Sujeitos: ['S01', 'S02', 'S04', 'S05', 'S06', 'S08', 'S09', 'S10', 'S13', 'S14', 'S15', 'S16', 'S17', 'S18', 'f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13']


Processa um sujeito de cada protocolo para validar o pipeline e ussa 'apply_labels' como protocolo'ANAEROBIC' ou 'AEROBIC' para selecionar as fases corretas automaticamente (V1/V2 detectado pelo ID).

In [ ]:
preprocessor = SignalPreprocessor()
extractor = FeatureExtractor(window_size=WINDOW_SIZE, step_size=STEP_SIZE)

for protocol in PROTOCOLS:
    protocol_dir = RAW_DATA_DIR / protocol
    if not protocol_dir.exists(): continue
    
    # Pega primeiro sujeito válido
    subjects = sorted([p.name for p in protocol_dir.iterdir() if p.is_dir()])
    test_subj = [s for s in subjects if s not in EXCLUSIONS.get(protocol, [])][0]
    subject_path = protocol_dir / test_subj
    
    print(f"=== {protocol} / {test_subj} ===")
    
    raw_data = load_subject_raw_data(subject_path)
    session_start = get_session_start_time(subject_path)
    tags_df = load_tags(subject_path, session_start)
    print(f"Sensores: {list(raw_data.keys())} | Tags: {len(tags_df)}")
    
    df_timeline = preprocessor.synchronize_data(raw_data)
    df_labeled = apply_labels(df_timeline, tags_df, subject_id=test_subj, protocol=protocol)
    
    counts = df_labeled['label'].value_counts()
    print(f"Repouso: {counts.get(0, 0)} | Exercício: {counts.get(1, 0)}")
    
    df_features = extractor.extract_features(raw_data, df_labeled)
    df_features = df_features.dropna()
    print(f"Features: {df_features.shape}")
    print(f"Labels: {df_features['label'].value_counts().to_dict()}")

=== ANAEROBIC / S01 ===
Sensores: ['ACC', 'BVP', 'EDA', 'TEMP', 'HR', 'IBI'] | Tags: 7
Repouso: 4117 | Exercício: 424
Features: (37, 51)
Labels: {0: 34, 1: 3}
=== AEROBIC / S01 ===
Sensores: ['ACC', 'BVP', 'EDA', 'TEMP', 'HR', 'IBI'] | Tags: 12
Repouso: 2113 | Exercício: 6012
Features: (60, 51)
Labels: {1: 45, 0: 15}


Processa todos os sujeitos válidos de cada protocolo e vai gerar um DataFrame por protocolo, com colunas 'subject_id' e 'protocol'.

In [ ]:
preprocessor = SignalPreprocessor()
extractor = FeatureExtractor(window_size=WINDOW_SIZE, step_size=STEP_SIZE)

results = {}

for protocol in PROTOCOLS:
    protocol_dir = RAW_DATA_DIR / protocol
    if not protocol_dir.exists(): continue
    
    subjects = sorted([p.name for p in protocol_dir.iterdir() if p.is_dir()])
    valid = [s for s in subjects if s not in EXCLUSIONS.get(protocol, [])]
    
    all_features = []
    failed = []
    
    for subj in tqdm(valid, desc=protocol):
        try:
            subject_path = protocol_dir / subj
            
            raw_data = load_subject_raw_data(subject_path)
            if not raw_data:
                failed.append((subj, "Dados vazios"))
                continue
            
            session_start = get_session_start_time(subject_path)
            tags_df = load_tags(subject_path, session_start)
            df_timeline = preprocessor.synchronize_data(raw_data)
            df_labeled = apply_labels(df_timeline, tags_df, subject_id=subj, protocol=protocol)
            
            df_feats = extractor.extract_features(raw_data, df_labeled)
            df_feats['subject_id'] = subj
            df_feats['protocol'] = protocol
            df_feats = df_feats.dropna()
            
            if not df_feats.empty:
                all_features.append(df_feats)
                ex = sum(df_feats['label'] == 1)
                rest = sum(df_feats['label'] == 0)
                print(f"  {subj}: {df_feats.shape[0]} janelas para Repouso = {rest} Exercicio = {ex}")
        
        except Exception as e:
            print(f"  ERRO {subj}: {e}")
            failed.append((subj, str(e)))
    
    results[protocol] = all_features
    print(f"\n{protocol}: {len(all_features)} sujeitos e falhas: {len(failed)}")
    if failed: print(f"  falhas: {failed}")

ANAEROBIC:   3%|▎         | 1/29 [00:02<00:59,  2.12s/it]

  S01: 37 janelas para Repouso = 34 Exercicio = 3


ANAEROBIC:   7%|▋         | 2/29 [00:04<00:56,  2.10s/it]

  S02: 36 janelas para Repouso = 34 Exercicio = 2


ANAEROBIC:  10%|█         | 3/29 [00:07<01:03,  2.45s/it]

...

  f11: 62 janelas para Repouso = 24 Exercicio = 38


AEROBIC:  96%|█████████▋| 26/27 [02:04<00:04,  4.34s/it]

  f12: 73 janelas para Repouso = 35 Exercicio = 38


AEROBIC: 100%|██████████| 27/27 [02:08<00:00,  4.75s/it]

  f13: 85 janelas para Repouso = 47 Exercicio = 38

AEROBIC: 27 sujeitos e falhas: 0


Salva um CSV por protocolo (dataset_anaerobic.csv, dataset_aerobic.csv) e esses CSVs são entrada para o Cenário 2 (AEROBIC vs ANAEROBIC) no script de modelagem, onde as janelas de exercício (label=1) de ambos são combinadas com rótulos distintos.

In [ ]:
meta_cols = ['subject_id', 'window_id', 'label', 'protocol']

for protocol in PROTOCOLS:
    if protocol not in results or not results[protocol]:
        print(f"{protocol}: sem dados")
        continue
    
    df = pd.concat(results[protocol], ignore_index=True)
    feature_cols = [c for c in df.columns if c not in meta_cols]
    df = df[[c for c in meta_cols if c in df.columns] + feature_cols]
    
    output_path = PROCESSED_DIR / f"dataset_{protocol.lower()}.csv"
    df.to_csv(output_path, index=False)
    
    print(f"{protocol}")
    print(f"Shape: {df.shape}")
    print(f"Sujeitos: {df['subject_id'].nunique()}")
    print(f"Labels: {df['label'].value_counts().to_dict()}")
    print(f"Salvo: {output_path}")

ANAEROBIC
Shape: (1544, 53)
Sujeitos: 29
Labels: {0: 1455, 1: 89}
Salvo: C:\DEV\mestrado\WSD\experiments\processed\dataset_anaerobic.csv
AEROBIC
Shape: (1967, 53)
Sujeitos: 27
Labels: {1: 1190, 0: 777}
Salvo: C:\DEV\mestrado\WSD\experiments\processed\dataset_aerobic.csv
